# Training a model

In [17]:
import torch
import transformers

from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, GPT2LMHeadModel, GPT2Tokenizer

load_dotenv()


# === load teacher model ===
model_name = "gpt2"

teacher_model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# btw `model = GPT2LMHeadModel.from_pretrained("gpt2")` does the same as
# model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

teacher_model.eval()

# === prepare new model ===
student_model = transformers.models.GPT2LMHeadModel(teacher_model.config)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
def generate_training_batch(batch_size=2, seq_len=30, warmup=2):
    data = teacher_model.generate(
        input_ids=None,
        attention_mask=None,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        max_new_tokens=seq_len + warmup,
        num_return_sequences=batch_size,
    )

    return tokenizer.decode(data[:, warmup + 1 :])


generate_training_batch()

['ing what some call \'work\' is hard for some people," she says. "And it goes back to your mental health. But it seems we',
 ' more complete list of the 5 most hated things in the world, check your local newspaper and online reviews section. But first and foremost, what does it']

In [64]:
# dataloader

from torch.utils.data import Dataset, DataLoader


def synthetic_batch_generator(batch_size=2, seq_len=20, num_batches=100):
    for _ in range(num_batches):
        tokens = teacher_model.generate(
            input_ids=None,
            attention_mask=None,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            max_new_tokens=seq_len - 1,
            num_return_sequences=batch_size,
        )
        attention_mask = torch.ones((batch_size, seq_len), dtype=torch.bool)

        yield tokens, attention_mask


for x, y in synthetic_batch_generator(num_batches=1):
    print(x)
    print(y)

tensor([[50256,    50, 35652,   468,   587,  1043,   287,  2279,   422, 15921,
           290, 14380,   284,  8237, 36656,    11, 22514,   290,    11,  3360],
        [50256,    12,  1400,   198,   198,    12,  1400,   198,   198,    12,
          1400,   198,   198,    12,  1400,   198,   198,    12,  1400,   198]])
tensor([[True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True, True, True,
         True, True, True, True, True, True, True, True]])


In [ ]:
# training loop

from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

# training parameters
batch_size = 2
seq_len = 20
num_batches = 4


# Move models to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
teacher_model.to(device)
student_model.to(device)

# Optimizer
optimizer = AdamW(student_model.parameters(), lr=5e-5)

# Loss function
loss_fn = CrossEntropyLoss()

# Training loop
iteration = 0
for inputs, attention_mask in synthetic_batch_generator(
    batch_size=batch_size, seq_len=seq_len, num_batches=num_batches
):
    iteration += 1

    inputs = inputs.to(device)
    attention_mask = attention_mask.to(device)

    # Forward pass: student model
    outputs = student_model(inputs, attention_mask=attention_mask, labels=inputs)
    loss = outputs.loss

    # Backward pass
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print(f"Epoch {iteration}, Loss: {loss.item()}")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 0, Loss: 10.783906936645508
Epoch 0, Loss: 10.662690162658691
Epoch 0, Loss: 10.606252670288086
Epoch 0, Loss: 10.5613431930542


In [51]:
x = model.generate(
    inputs=None,
    do_sample=True,
    temperature=0.7,
    max_new_tokens=30,
    num_return_sequences=3,
)
tokenizer.decode(x)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


['<|endoftext|>"I\'m sure it\'s been a long time since I\'ve read this book, but here\'s a little bit of advice for you young ladies reading',
 '<|endoftext|>I had a lot of fun watching the episode with all the different characters being played by different actors, so I thought it would be a great opportunity to',
 "<|endoftext|>\nBy Scott Adams\n\nThe Green Bay Packers' current quarterback, Matt Flynn, is making his NFL debut against the Carolina Panthers on Saturday.\n"]

In [22]:
type(new_gpt2)

transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel

In [23]:
text = "blablab"
input = tokenizer(text, return_tensors="pt")
input


model(**input)

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[-30.3457, -29.5554, -32.3682,  ..., -38.2069, -36.7786, -30.5667],
         [-83.5400, -81.4145, -85.9746,  ..., -89.4802, -90.8593, -82.4991],
         [-81.5572, -81.6088, -84.2322,  ..., -93.0027, -89.4398, -81.0299]]],
       grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)

In [ ]:
import transformers

transformers.models.GPT2PreTrainedModel


class Controlen(transformers.models.GPT2PreTrainedModel):
    def __init__(self, config):
        super.__init__(config)

        self.embod_dim = config.hidden_size

        self.h = transformers.models.gpt2.modeling_gpt2.GPT2Block

In [6]:
import transformers

# def create_causal_mask(
#     config: PreTrainedConfig,
#     inputs_embeds: torch.Tensor,
#     attention_mask: torch.Tensor | None,
#     cache_position: torch.Tensor,
#     past_key_values: Cache | None,
#     position_ids: torch.Tensor | None = None,
#     or_mask_function: Callable | None = None,
#     and_mask_function: Callable | None = None,
# ) -> torch.Tensor | BlockMask | None:

transformers.masking_utils.create_causal_mask

<function transformers.masking_utils.create_causal_mask(config: transformers.configuration_utils.PreTrainedConfig, inputs_embeds: torch.Tensor, attention_mask: torch.Tensor | None, cache_position: torch.Tensor, past_key_values: transformers.cache_utils.Cache | None, position_ids: torch.Tensor | None = None, or_mask_function: collections.abc.Callable | None = None, and_mask_function: collections.abc.Callable | None = None) -> torch.Tensor | torch.nn.attention.flex_attention.BlockMask | None>

In [2]:
# Define a new bidirectional transformer block
class BidirectionalTransformerBlock(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn = torch.nn.MultiheadAttention(
            config.n_embd, config.n_head, dropout=config.attn_pdrop
        )
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(
                config.n_embd,
                config.n_inner if config.n_inner is not None else 4 * config.n_embd,
            ),
            torch.nn.GELU(),
            torch.nn.Linear(
                config.n_inner if config.n_inner is not None else 4 * config.n_embd,
                config.n_embd,
            ),
        )

    def forward(self, hidden_states, attention_mask=None):
        # Bidirectional attention: no causal mask
        attn_output, _ = self.attn(
            hidden_states,
            hidden_states,
            hidden_states,
            need_weights=False,
            attn_mask=None if attention_mask is None else ~attention_mask,
        )
        hidden_states = hidden_states + attn_output
        hidden_states = hidden_states + self.mlp(hidden_states)
        return hidden_states

In [ ]:
layers = model.transformer.h
split_point = len(layers) // 2

upper_layers = torch.nn.ModuleList(layers[split_point:])

new_bidirectional_block = BidirectionalTransformerBlock(model.config)

new_lower_layers = torch.nn.ModuleList(
    [BidirectionalTransformerBlock(model.config) for _ in range(split_point)]
)

new_model = GPT2LMHeadModel(model.config)
new_model.transformer.h = torch.nn.ModuleList(
    list(new_lower_layers) + list(upper_layers)
)

input_ids = torch.randint(0, 50000, (1, 10))
outputs = new_model(input_ids)
print(outputs.logits.shape)

In [21]:
new_bidirectional_block

BidirectionalTransformerBlock(
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (mlp): Sequential(
    (0): Linear(in_features=768, out_features=3072, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=3072, out_features=768, bias=True)
  )
)

In [23]:
batch_size = 2
seq_len = 5
n_hidden = 768
x = torch.rand((batch_size, seq_len, n_hidden))
new_bidirectional_block(x).shape

torch.Size([2, 5, 768])

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Config

# Load the GPT2LMHeadModel
model = GPT2LMHeadModel.from_pretrained("gpt2")

In [ ]:
import copy
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel


class SplitGPT2(nn.Module):
    def __init__(self, name_or_path="gpt2"):
        super().__init__()
        base = GPT2LMHeadModel.from_pretrained(name_or_path)
        tr = base.transformer
        layers = tr.h
        split_point = len(layers) // 2

        # Trainable lower part
        self.token_emb = nn.Embedding(tr.wte.num_embeddings, tr.wte.embedding_dim)
        self.token_emb.weight.data.copy_(tr.wte.weight.data)

        self.pos_emb = copy.deepcopy(
            tr.wpe
        )  # keep it simple; you can also make this trainable
        self.drop = copy.deepcopy(tr.drop)

        # One trainable block replacing the lower half
        self.lower_block = copy.deepcopy(layers[split_point - 1])

        # Frozen upper half
        self.upper_layers = nn.ModuleList(
            copy.deepcopy(layer) for layer in layers[split_point:]
        )
        self.ln_f = copy.deepcopy(tr.ln_f)

        self.config = tr.config

        # Freeze everything
        for p in self.parameters():
            p.requires_grad = False

        # Unfreeze the new lower part
        for p in self.token_emb.parameters():
            p.requires_grad = True
        for p in self.lower_block.parameters():
            p.requires_grad = True

    def _make_attention_mask(self, attention_mask, hidden_states):
        """
        attention_mask: (batch, seq_len), 1 for real tokens, 0 for padding
        Returns additive mask of shape (batch, 1, 1, seq_len)
        """
        return attention_mask

    def lower_forward(self, input_ids, attention_mask=None, position_ids=None):
        """
        Query the lower part independently.

        input_ids: (batch, seq_len)
        returns: (batch, seq_len, d_model)
        """
        batch_size, seq_len = input_ids.shape

        if position_ids is None:
            position_ids = torch.arange(seq_len, device=input_ids.device)
            position_ids = position_ids.unsqueeze(0).expand(batch_size, -1)

        hidden_states = self.token_emb(input_ids) + self.pos_emb(position_ids)
        hidden_states = self.drop(hidden_states)

        attn_mask = self._make_attention_mask(attention_mask, hidden_states)

        print("hidden_states:", hidden_states.shape)
        print("attn_mask:", attn_mask.shape)
        hidden_states = self.lower_block(
            hidden_states,
            layer_past=None,
            attention_mask=attn_mask,
            head_mask=None,
            use_cache=False,
            output_attentions=False,
        )[0]

        return hidden_states

    def forward(self, input_ids, attention_mask=None, position_ids=None):
        """
        Full model forward.

        input_ids: (batch, seq_len)
        returns: (batch, seq_len, d_model)
        """
        hidden_states = self.lower_forward(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
        )

        attn_mask = self._make_attention_mask(attention_mask, hidden_states)

        for block in self.upper_layers:
            hidden_states = block(
                hidden_states,
                layer_past=None,
                attention_mask=attn_mask,
                head_mask=None,
                use_cache=False,
                output_attentions=False,
            )[0]

        hidden_states = self.ln_f(hidden_states)
        return hidden_states

In [ ]:
model = SplitGPT2("gpt2")

input_ids = torch.tensor(
    [
        [15496, 995, 50256, 50256],  # padded
        [1212, 318, 257, 1332],  # full length
    ]
)

attention_mask = torch.tensor(
    [
        [1, 1, 0, 0],
        [1, 1, 1, 1],
    ],
    dtype=torch.bool,
)

lower = model.lower_forward(input_ids, attention_mask=attention_mask)
# full = model(input_ids, attention_mask=attention_mask)

print(lower.shape)  # (2, 4, 768)
# print(full.shape)  # (2, 4, 768)

In [36]:
base = GPT2LMHeadModel.from_pretrained("gpt2")
tr = base.transformer
layers = tr.h
split_point = len(layers) // 2

# Trainable lower part
token_emb = nn.Embedding(tr.wte.num_embeddings, tr.wte.embedding_dim)
token_emb.weight.data.copy_(tr.wte.weight.data)

pos_emb = copy.deepcopy(tr.wpe)  # keep it simple; you can also make this trainable
drop = copy.deepcopy(tr.drop)

# One trainable block replacing the lower half
lower_block = copy.deepcopy(layers[split_point - 1])

input_ids = torch.tensor(
    [
        [15496, 995, 50256, 50256],  # padded
        [1212, 318, 257, 1332],  # full length
    ]
)

attention_mask = torch.tensor(
    [
        [1, 1, 0, 0],
        [1, 1, 1, 1],
    ],
    dtype=torch.bool,
)
attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)

batch_size, seq_len = input_ids.shape

position_ids = torch.arange(seq_len, device=input_ids.device)
position_ids = position_ids.unsqueeze(0).expand(batch_size, -1)

hidden_states = token_emb(input_ids) + pos_emb(position_ids)
hidden_states = drop(hidden_states)


print("hidden_states:", hidden_states.shape)
print("attn_mask:", attention_mask.shape)
hidden_states = lower_block(
    hidden_states,
    layer_past=None,
    attention_mask=attention_mask,
    head_mask=None,
    use_cache=False,
    output_attentions=False,
)[0]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

hidden_states: torch.Size([2, 4, 768])
attn_mask: torch.Size([2, 1, 1, 4])
